In [4]:
!pip install jupyter selenium webdriver_manager

import sys
!{sys.executable} -m pip install --upgrade pip

# Automatically download and install the appropriate ChromeDriver
from webdriver_manager.chrome import ChromeDriverManager
!{sys.executable} -m pip install --upgrade webdriver_manager

# Install the ChromeDriver
!{sys.executable} -m webdriver_manager.chrome install

print("All necessary packages have been installed.")

All necessary packages have been installed.


In [5]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import csv
from webdriver_manager.chrome import ChromeDriverManager

In [6]:
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')  # Run in background
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

driver = setup_driver()

In [86]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

def create_webdriver_session(headless=False):
    try:
        options = Options()
        if headless:
            options.add_argument('--headless')
        
        # Add additional options to make the browser more stable
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        # Create a new ChromeDriver service
        service = Service(ChromeDriverManager().install())
        
        # Create a new driver instance
        driver = webdriver.Chrome(service=service, options=options)
        
        print("WebDriver session created successfully.")
        return driver
    except Exception as e:
        print(f"An error occurred while creating the WebDriver session: {str(e)}")
        return None

# Usage
driver = create_webdriver_session()
if driver is None:
    print("Failed to create WebDriver session. Please check your Chrome and ChromeDriver installations.")
    exit(1)

WebDriver session created successfully.


In [87]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time

def check_login_status(driver):
    driver.get("https://www.crunchbase.com/home")
    time.sleep(5)  # Wait for page to load
    print("Current URL:", driver.current_url)
    print("Page title:", driver.title)
    
    if "home" in driver.current_url and "Activity Feed" in driver.title:
        print("User appears to be already logged in.")
        return True
    return False

def login_to_crunchbase(driver, username, password, max_retries=3):
    if check_login_status(driver):
        return True

    for attempt in range(max_retries):
        try:
            print(f"\nLogin attempt {attempt + 1}")
            driver.get("https://www.crunchbase.com/login")
            time.sleep(5)  # Wait for page to load
            
            print("Current URL:", driver.current_url)
            print("Page title:", driver.title)
            
            # Step 1: Locate and fill the email field
            print("Attempting to locate email field...")
            email_field = WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='email']"))
            )
            print("Email field found.")
            email_field.clear()
            email_field.send_keys(username)
            print("Email entered:", username)
            
            # Step 2: Locate and fill the password field
            print("Attempting to locate password field...")
            password_field = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='password']"))
            )
            print("Password field found.")
            password_field.clear()
            password_field.send_keys(password)
            print("Password entered.")
            
            # Step 3: Locate and click the login button
            print("Attempting to locate login button...")
            login_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "button[type='submit']"))
            )
            print("Login button found.")
            login_button.click()
            print("Login button clicked.")
            
            # Step 4: Wait for page change
            print("Waiting for login response...")
            time.sleep(5)  # Wait for potential redirects
            
            if check_login_status(driver):
                print("Login successful.")
                return True
            
            print("Login unsuccessful. Checking for error messages...")
            try:
                error_message = WebDriverWait(driver, 5).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, ".errors, .alert-error"))
                )
                print("Login error detected:", error_message.text)
            except TimeoutException:
                print("No error message found on the page.")
            
        except Exception as e:
            print(f"Error during login attempt {attempt + 1}: {str(e)}")
        
        if attempt < max_retries - 1:
            print("Retrying login...")
            time.sleep(5)  # Wait before retrying
        else:
            print("Max login attempts reached. Login failed.")
    
    return False

# Usage
username = "manishkhakhal@gmail.com"
password = "GEvbZxE2&X?ddPj"
login_successful = login_to_crunchbase(driver, username, password)

if login_successful:
    print("Login successful or already logged in. Proceeding with scraping.")
else:
    print("Login failed, please check your credentials and try again.")

Current URL: https://www.crunchbase.com/home
Page title: Just a moment...

Login attempt 1
Current URL: https://www.crunchbase.com/login
Page title: Just a moment...
Attempting to locate email field...
Error during login attempt 1: Message: 
Stacktrace:
0   chromedriver                        0x00000001047e39d8 cxxbridge1$str$ptr + 1887096
1   chromedriver                        0x00000001047dbe40 cxxbridge1$str$ptr + 1855456
2   chromedriver                        0x00000001043e0be0 cxxbridge1$string$len + 89508
3   chromedriver                        0x0000000104424f10 cxxbridge1$string$len + 368852
4   chromedriver                        0x000000010445e4a4 cxxbridge1$string$len + 603752
5   chromedriver                        0x0000000104419a08 cxxbridge1$string$len + 322508
6   chromedriver                        0x000000010441a66c cxxbridge1$string$len + 325680
7   chromedriver                        0x00000001047aa098 cxxbridge1$str$ptr + 1651256
8   chromedriver                 

In [34]:

username = "manishkhakhal@gmail.com"
password = "GEvbZxE2&X?ddPj"


Login attempt 1
Already logged in
Proceeding with scraping...


In [46]:
if verify_crunchbase_access(driver, "apple"):
    print("Crunchbase access verified. Ready to proceed with scraping.")
else:
    print("Unable to verify Crunchbase access. Please check your account permissions.")
    print("\nDiagnostic Information:")
    print(f"Current URL: {driver.current_url}")
    print(f"Page Title: {driver.title}")
    print("Page Source Snippet:")
    print(driver.page_source[:1000])  # Print the first 1000 characters of the page source

Successfully accessed apple's Crunchbase page.
Crunchbase access verified. Ready to proceed with scraping.


In [50]:
def verify_crunchbase_login_general(driver):
    # Navigate to the home page
    driver.get("https://www.crunchbase.com/home")
    
    # Wait for the page to load
    import time
    time.sleep(5)
    
    # Check the current URL
    current_url = driver.current_url
    print(f"Current URL: {current_url}")
    
    # Check the page title
    page_title = driver.title
    print(f"Page title: {page_title}")
    
    # Check for login-specific content
    page_source = driver.page_source.lower()
    logged_in_indicators = ["activity feed", "my lists", "recents", "following"]
    logged_out_indicators = ["log in", "sign up", "join crunchbase"]
    
    logged_in_found = any(indicator in page_source for indicator in logged_in_indicators)
    logged_out_found = any(indicator in page_source for indicator in logged_out_indicators)
    
    if logged_in_found and not logged_out_found:
        print("Login verified: Found logged-in indicators and no logged-out indicators.")
        return True
    elif logged_out_found and not logged_in_found:
        print("Not logged in: Found logged-out indicators and no logged-in indicators.")
        return False
    else:
        print("Login status unclear. Found conflicting or no clear indicators.")
        return None

# Usage
login_status = verify_crunchbase_login_general(driver)
if login_status:
    print("Logged in. Proceeding with scraping.")
elif login_status is False:
    print("Not logged in. Please log in and try again.")
else:
    print("Unable to determine login status. Please check manually.")

Current URL: https://www.crunchbase.com/home
Page title: Activity Feed | Crunchbase
Login verified: Found logged-in indicators and no logged-out indicators.
Logged in. Proceeding with scraping.


In [48]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

def verify_crunchbase_login(driver, timeout=10):
    try:
        # Check for elements that should only be visible when logged in
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "[data-test='user-menu']"))
        )
        print("Login verified: User menu found.")
        return True
    except TimeoutException:
        print("Login status unclear: User menu not found.")
        
        # Check if we're on the login page
        if "login" in driver.current_url.lower():
            print("Not logged in: On login page.")
            return False
        
        print("Login status unclear. Please check manually.")
        return None

# Usage
login_status = verify_crunchbase_login(driver)
if login_status:
    print("Proceeding with scraping.")
elif login_status is False:
    print("Need to log in again.")
else:
    print("Unable to determine login status. Please check manually.")

Login status unclear: User menu not found.
Login status unclear. Please check manually.
Unable to determine login status. Please check manually.


In [91]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time

def get_company_investors(driver, company_name, max_retries=3, timeout=60):
    url = f"https://www.crunchbase.com/organization/{company_name}/company_financials"
    for attempt in range(max_retries):
        try:
            print(f"\nAttempt {attempt + 1} to load investors for {company_name}")
            driver.get(url)
            print(f"Current URL: {driver.current_url}")
            
            # Wait for the page to load
            WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located((By.ID, "investors"))
            )
            
            # Find all investor rows
            investor_rows = driver.find_elements(By.CSS_SELECTOR, "#investors tbody tr")
            
            investors = []
            for row in investor_rows:
                try:
                    investor_name = row.find_element(By.CSS_SELECTOR, "td:nth-child(1)").text.strip()
                    is_lead = row.find_element(By.CSS_SELECTOR, "td:nth-child(2)").text.strip()
                    funding_round = row.find_element(By.CSS_SELECTOR, "td:nth-child(3)").text.strip()
                    partners = row.find_element(By.CSS_SELECTOR, "td:nth-child(4)").text.strip()
                    
                    investor_info = {
                        "name": investor_name,
                        "is_lead": is_lead == "Yes",
                        "funding_round": funding_round,
                        "partners": partners if partners != "—" else None
                    }
                    investors.append(investor_info)
                except NoSuchElementException:
                    continue  # Skip rows that don't have the expected structure
            
            print(f"Found {len(investors)} investors for {company_name}")
            return investors
        
        except TimeoutException:
            print(f"Timeout: Could not load investors for {company_name}")
        except Exception as e:
            print(f"An error occurred: {str(e)}")
        
        if attempt < max_retries - 1:
            print(f"Retrying in 5 seconds... (Attempt {attempt + 2} of {max_retries})")
            time.sleep(5)
        else:
            print("Max retries reached. Unable to load investors.")
    
    print("Returning empty list of investors.")
    return []

# Usage
plentify_investors = get_company_investors(driver, "plentify")
for investor in plentify_investors:
    print(f"Investor: {investor['name']}")
    print(f"  Lead Investor: {'Yes' if investor['is_lead'] else 'No'}")
    print(f"  Funding Round: {investor['funding_round']}")
    print(f"  Partners: {investor['partners'] if investor['partners'] else 'None'}")
    print()


Attempt 1 to load investors for plentify
Current URL: https://www.crunchbase.com/organization/plentify/company_financials
Investors section found.
No investor elements found. Printing page source for debugging:
<html lang="en"><head supported-browser="true" sentry-dsn="https://ad6d278f98e347409c82d6e3899597a4@sentry.io/190044" perimeterx-id="rw7M6iAV"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin="">
<title>Plentify - Funding, Financials, Valuation &amp; Investors</title>
<meta name="viewport" content="width=device-width, minimum-scale=1.0, initial-scale=1.0">
<meta name="google-site-verification" content="D0Crp7AShYxB6XBQLfVq4hOgCHyZ_qQiMOTpUioaeQU">
<base href="/">
<link rel="icon" href="/favicon.ico?v=2.0">
<link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png?v=2.0">
<link rel="icon" type="image/png" sizes="32x32" href="/favicon-32x32.png?v=2.0">
<link rel="icon" type="image/png" sizes="16x16" href="/favicon-16x16.png?v=2.0">
<link rel="man

In [78]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

def analyze_investors_section(driver, timeout=30):
    try:
        # Wait for the investors section to be present
        investors_section = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.ID, "investors"))
        )
        print("Investors section found. Analyzing content...")

        # Get all child elements of the investors section
        children = investors_section.find_elements(By.XPATH, ".//*")
        
        print(f"Total elements in investors section: {len(children)}")
        
        # Analyze the first few elements
        for i, child in enumerate(children[:10]):  # Analyze first 10 elements
            print(f"\nElement {i+1}:")
            print(f"  Tag: {child.tag_name}")
            print(f"  Classes: {child.get_attribute('class')}")
            print(f"  Text content: {child.text[:100]}...")  # First 100 characters of text
        
        # Check for specific indicators
        paywall_indicators = investors_section.find_elements(By.XPATH, ".//*[contains(@class, 'paywall') or contains(@class, 'premium') or contains(@class, 'upgrade')]")
        if paywall_indicators:
            print("\nPaywall or premium content indicators found!")
            for indicator in paywall_indicators:
                print(f"  {indicator.tag_name}: {indicator.get_attribute('class')}")
        
        # Check for any buttons or links
        buttons = investors_section.find_elements(By.XPATH, ".//button | .//a")
        if buttons:
            print("\nButtons or links found in the investors section:")
            for button in buttons:
                print(f"  {button.tag_name}: {button.text} (Class: {button.get_attribute('class')})")
        
        return True
    except TimeoutException:
        print("Timed out waiting for investors section to load.")
        return False
    except Exception as e:
        print(f"An error occurred while analyzing the investors section: {str(e)}")
        return False

# Usage
analyze_investors_section(driver)

Investors section found. Analyzing content...
Total elements in investors section: 0


True

In [81]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import time

def wait_for_dynamic_content(driver, timeout=60):
    print("Waiting for dynamic content to load...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        investors_section = driver.find_element(By.ID, "investors")
        if len(investors_section.find_elements(By.XPATH, ".//*")) > 0:
            print(f"Content found after {time.time() - start_time:.2f} seconds.")
            return True
        time.sleep(1)
    print(f"No content found after {timeout} seconds.")
    return False

# Usage
if wait_for_dynamic_content(driver):
    print("Dynamic content loaded successfully.")
else:
    print("Failed to load dynamic content.")

Waiting for dynamic content to load...


WebDriverException: Message: disconnected: not connected to DevTools
  (failed to check if window was closed: disconnected: not connected to DevTools)
  (Session info: chrome=128.0.6613.86)
Stacktrace:
0   chromedriver                        0x00000001030179d8 cxxbridge1$str$ptr + 1887096
1   chromedriver                        0x000000010300fe40 cxxbridge1$str$ptr + 1855456
2   chromedriver                        0x0000000102c14be0 cxxbridge1$string$len + 89508
3   chromedriver                        0x0000000102bfea68 core::str::slice_error_fail::h6c488016ada29016 + 64036
4   chromedriver                        0x0000000102bfe9a8 core::str::slice_error_fail::h6c488016ada29016 + 63844
5   chromedriver                        0x0000000102c91f08 cxxbridge1$string$len + 602316
6   chromedriver                        0x0000000102c4da08 cxxbridge1$string$len + 322508
7   chromedriver                        0x0000000102c4e66c cxxbridge1$string$len + 325680
8   chromedriver                   

In [90]:
from selenium.webdriver.common.by import By
from collections import Counter

def extract_page_structure(driver):
    print("Extracting page structure...")
    
    # Get all elements
    elements = driver.find_elements(By.XPATH, "//*")
    
    # Extract classes and IDs
    classes = []
    ids = []
    for element in elements:
        class_attr = element.get_attribute("class")
        if class_attr:
            classes.extend(class_attr.split())
        id_attr = element.get_attribute("id")
        if id_attr:
            ids.append(id_attr)
    
    # Count occurrences
    class_counts = Counter(classes)
    id_counts = Counter(ids)
    
    print(f"\nFound {len(elements)} elements.")
    print(f"Found {len(class_counts)} unique classes and {len(id_counts)} unique IDs.")
    
    print("\nMost common classes:")
    for class_name, count in class_counts.most_common(10):
        print(f"  {class_name}: {count}")
    
    print("\nAll IDs found:")
    for id_name, count in id_counts.items():
        print(f"  {id_name}: {count}")
    
    # Check for specific classes or IDs related to investors or financial information
    investor_related = [item for item in classes + ids if "investor" in item.lower() or "financial" in item.lower()]
    if investor_related:
        print("\nInvestor or financial related classes/IDs found:")
        for item in investor_related:
            print(f"  {item}")
    else:
        print("\nNo investor or financial related classes/IDs found.")
# Usage
extract_page_structure(driver)

Extracting page structure...

Found 1776 elements.
Found 265 unique classes and 43 unique IDs.

Most common classes:
  ng-star-inserted: 570
  accent: 76
  component--field-formatter: 68
  mat-mdc-focus-indicator: 48
  mat-mdc-tooltip-trigger: 38
  mat-mdc-button-base: 36
  mat-mdc-button-persistent-ripple: 36
  mat-mdc-button-touch-target: 36
  inherit: 34
  default: 31

All IDs found:
  tb_tfa_script: 1
  demandbase_js_lib: 1
  onetrust-style: 1
  _pendo-css_: 1
  search-crunchbase: 1
  mat-tab-link-1: 1
  mat-tab-link-2: 1
  mat-tab-link-3: 1
  mat-tab-link-4: 1
  mat-tab-link-5: 1
  mat-tab-link-6: 1
  mat-tab-nav-panel-0: 1
  company_financials_highlights: 1
  company_financials_about: 1
  funding_rounds: 1
  investors: 1
  cdk-step-content-0-0: 1
  cdk-step-content-0-1: 1
  ng-state: 1
  cdk-live-announcer-0: 1
  onetrust-consent-sdk: 1
  cdk-describedby-message-ng-1-2: 1
  cdk-describedby-message-ng-1-3: 1
  cdk-describedby-message-ng-1-4: 1
  cdk-describedby-message-ng-1-5: 1
 

In [52]:
# First, verify login status
login_status = verify_crunchbase_login_general(driver)
if login_status:
    print("Logged in. Proceeding with scraping.")
    
    # Now, let's try to get investors for Plentify
    plentify_investors = get_company_investors(driver, "plentify")
    print(f"Investors for Plentify: {plentify_investors}")
elif login_status is False:
    print("Not logged in. Please log in and try again.")
else:
    print("Unable to determine login status. Please check manually.")

Current URL: https://www.crunchbase.com/home
Page title: Activity Feed | Crunchbase
Login verified: Found logged-in indicators and no logged-out indicators.
Logged in. Proceeding with scraping.
Current URL: https://www.crunchbase.com/home
Page title: Activity Feed | Crunchbase
Login verified: Found logged-in indicators and no logged-out indicators.

Attempt 1 to load investors for plentify
Current URL: https://www.crunchbase.com/organization/plentify/company_financials
Investor section found using selector: #investors
Found 0 investors for plentify
Investors for Plentify: []


In [7]:
driver.quit()